# Punto 3 · IA Aplicada con LLM 
## Data & AI Challenge · Insurance Claims Fraud Detection

**Autor:** Santiago Pereyra

## Caso de uso elegido: Auditor Automático de Siniestros Sospechosos

### Qué problema resuelve
En operaciones de siniestros hay muchas señales dispersas: demora en el reporte, monto elevado, severidad alta, ausencia de documentación, vendor faltante, incidentes nocturnos, pólizas recientes, etc.  
Esas señales existen en los datos, pero no llegan convertidas en una **narrativa clara y accionable** para el analista o supervisor.

### Qué aporta el LLM
El LLM toma un registro estructurado y devuelve una **nota de auditoría** en lenguaje natural con formato consistente:
- resumen ejecutivo del caso,
- señales de riesgo detectadas,
- vacíos de evidencia / documentación faltante,
- recomendación operativa inmediata,
- prioridad sugerida.

### Qué NO hace este notebook
- No aprueba ni rechaza siniestros automáticamente.
- No reemplaza criterios legales, actuariales o antifraude.
- No intenta probar fraude.
- No se apoya solo en el LLM: primero construye señales estructuradas y luego usa el LLM para **explicar y priorizar**.

### Conexión con el Punto 2
En el benchmark predictivo (Punto 2) se demostró que la información estructurada sola tiene capacidad limitada para explicar el rechazo. Esto indica que la decisión depende de información no estructurada (narrativas, documentación, juicio del ajustador) — exactamente donde un LLM agrega valor.

---


# 0. Setup

In [1]:
# =========================
# Instalación (ejecutar solo una vez)
# =========================
# %pip install openai pandas pyarrow duckdb python-dotenv

print("\n Dependencias listas")



 Dependencias listas


In [2]:
# =========================
# Imports
# =========================
import warnings
warnings.filterwarnings("ignore")

import os
import json
import zipfile
from pathlib import Path
from textwrap import dedent

import numpy as np
import pandas as pd
import duckdb

# --- SDK de OpenAI ---
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
    print("\n openai SDK disponible")
except ImportError:
    OPENAI_AVAILABLE = False
    print("\n SDK openai no instalado. Se usará demo offline.")

# --- dotenv para cargar API key desde archivo .env ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # No es crítico

# Configuración del cliente
API_KEY = os.environ.get("OPENAI_API_KEY", None)
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")  # Modelo económico y rápido

if API_KEY and OPENAI_AVAILABLE:
    openai_client = OpenAI(api_key=API_KEY)
    USE_REAL_API = True
    print(f"\n API key detectada. Modelo: {MODEL_NAME}")
else:
    USE_REAL_API = False
    print("\n  Sin API key. La demo correrá con simulación offline.")
    print("   Para usar la API real:")
    print("   - PowerShell: $env:OPENAI_API_KEY='sk-...'")
    print("   - Linux/Mac:  export OPENAI_API_KEY='sk-...'")
    print("   - O crear archivo .env con: OPENAI_API_KEY=sk-...")



 openai SDK disponible

 API key detectada. Modelo: gpt-4o-mini


# 1. Carga de la base analítica

In [3]:
# =========================
# Rutas del proyecto (consistentes con Punto 1)
# =========================
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / "Desafio_1_limpieza" / "data"
SILVER_DIR   = DATA_DIR / "silver"
GOLD_DIR     = DATA_DIR / "gold"
BRONZE_DIR   = DATA_DIR / "bronze"
DB_DIR       = PROJECT_DIR / "db"
DUCKDB_PATH  = DB_DIR / "tekne_claims.duckdb"
OUTPUT_DIR   = PROJECT_DIR / "Desafio_3_IA" / "outputs_llm"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_DIR.mkdir(parents=True, exist_ok=True)

def load_analytic_base() -> pd.DataFrame:
    """Carga la base analítica con fallback en 3 niveles."""
    gold_path   = GOLD_DIR / "claims_enriched_gold.parquet"
    silver_path = SILVER_DIR / "claims_silver.parquet"
    zip_path    = PROJECT_DIR / "archive.zip"

    if gold_path.exists():
        print(f"\n Cargando desde GOLD: {gold_path.name}")
        return pd.read_parquet(gold_path)
    elif silver_path.exists():
        print(f"\n Cargando desde SILVER: {silver_path.name}")
        return pd.read_parquet(silver_path)
    elif zip_path.exists():
        print("\n Reconstruyendo desde ZIP...")
        with zipfile.ZipFile(zip_path) as zf:
            claims = pd.read_csv(zf.open("insurance_data.csv"))
        claims.columns = [c.lower() for c in claims.columns]
        for col in ["txn_date_time", "policy_eff_dt", "loss_dt", "report_dt"]:
            claims[col] = pd.to_datetime(claims[col], errors="coerce")
        return claims
    else:
        raise FileNotFoundError("Ejecutar Punto 1 primero o colocar archive.zip en la raíz.")

claims_df = load_analytic_base()
claims_df.columns = [c.strip().lower() for c in claims_df.columns]

# Asegurar columnas derivadas
delay_col = next((c for c in ["days_report_delay", "report_delay_days"] if c in claims_df.columns), None)
if delay_col and delay_col != "days_report_delay":
    claims_df["days_report_delay"] = claims_df[delay_col]
if "days_report_delay" not in claims_df.columns:
    claims_df["days_report_delay"] = (claims_df["report_dt"] - claims_df["loss_dt"]).dt.days

if "loss_ratio" not in claims_df.columns and "claim_to_premium_ratio" not in claims_df.columns:
    claims_df["claim_to_premium_ratio"] = np.where(
        claims_df["premium_amount"].fillna(0) > 0,
        claims_df["claim_amount"] / claims_df["premium_amount"], np.nan
    )
elif "loss_ratio" in claims_df.columns and "claim_to_premium_ratio" not in claims_df.columns:
    claims_df["claim_to_premium_ratio"] = claims_df["loss_ratio"]

if "has_vendor" not in claims_df.columns:
    claims_df["has_vendor"] = claims_df["vendor_id"].notna().astype(int)

# Policy age
if "policy_age_days_at_loss" not in claims_df.columns and "policy_age_at_loss_days" not in claims_df.columns:
    claims_df["policy_age_days_at_loss"] = (claims_df["loss_dt"] - claims_df["policy_eff_dt"]).dt.days

print(f"Base analítica: {claims_df.shape[0]:,} filas × {claims_df.shape[1]} columnas")



 Cargando desde GOLD: claims_enriched_gold.parquet
Base analítica: 10,000 filas × 64 columnas


# 2. Señales estructuradas y score heurístico

Antes de usar el LLM, se construye una capa de **señales transparentes e interpretables**.  
El score heurístico es **ponderado** (no binario): cada flag tiene un peso que refleja su relevancia relativa.

No es un score productivo ni antifraude formal — es una herramienta de screening para priorizar revisión.


In [18]:
# =========================
# Cálculo de flags y score ponderado
# =========================

def compute_suspicion_score(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula flags de riesgo y un score ponderado por claim."""
    df = df.copy()
    
    # Resolver nombres de columnas (compatible con silver y gold)
    ratio_col = "claim_to_premium_ratio" if "claim_to_premium_ratio" in df.columns else "loss_ratio"
    delay_col = "days_report_delay" if "days_report_delay" in df.columns else "report_delay_days"
    age_col = next((c for c in ["policy_age_days_at_loss", "policy_age_at_loss_days"] if c in df.columns), None)
    
    # --- 8 flags de riesgo ---
    # Flag 1: Loss ratio extremo (peso 15)
    df["flag_high_ratio"] = (df[ratio_col].fillna(0) > 500).astype(int)
    
    # Flag 2: Reporte el mismo día (peso 10)
    df["flag_same_day_report"] = (df[delay_col].fillna(99) == 0).astype(int)
    
    # Flag 3: Total Loss sin reporte policial (peso 20)
    df["flag_total_loss_no_police"] = (
        (df["incident_severity"] == "Total Loss") & (df["police_report_available"] == 0)
    ).astype(int)
    
    # Flag 4: Sin vendor investigador (peso 10)
    df["flag_vendor_missing"] = (df["has_vendor"] == 0).astype(int)
    
    # Flag 5: Incidente en otro estado (peso 5)
    if "incident_state" in df.columns and "state" in df.columns:
        df["flag_cross_state"] = (df["state"] != df["incident_state"]).astype(int)
    else:
        df["flag_cross_state"] = 0
    
    # Flag 6: Incidente nocturno (0-5 hs) con severidad alta (peso 10)
    df["flag_night_high_severity"] = (
        df["incident_hour_of_the_day"].between(0, 5) & 
        (df["incident_severity"].isin(["Total Loss", "Major Loss"]))
    ).astype(int)
    
    # Flag 7: Póliza reciente < 30 días (peso 15)
    if age_col:
        df["flag_recent_policy"] = (df[age_col].fillna(999999) <= 30).astype(int)
    else:
        df["flag_recent_policy"] = 0
    
    # Flag 8: Severidad alta sin autoridad contactada (peso 10)
    df["flag_no_authority_severe"] = (
        (df["incident_severity"].isin(["Total Loss", "Major Loss"])) &
        (df["authority_contacted"].isin(["None", pd.NA, None]) | df["authority_contacted"].isna())
    ).astype(int)
    
    # --- Score ponderado ---
    df["suspicion_score"] = (
        df["flag_high_ratio"] * 15
        + df["flag_same_day_report"] * 10
        + df["flag_total_loss_no_police"] * 20
        + df["flag_vendor_missing"] * 10
        + df["flag_cross_state"] * 5
        + df["flag_night_high_severity"] * 10
        + df["flag_recent_policy"] * 15
        + df["flag_no_authority_severe"] * 10
    )
    
    return df

claims_scored = compute_suspicion_score(claims_df)

print("=== DISTRIBUCIÓN DEL SCORE PONDERADO ===")
print(claims_scored["suspicion_score"].describe())
print(f"\nClaims con score ≥ 30: {(claims_scored['suspicion_score'] >= 30).sum()}")
print(f"Claims con score ≥ 50: {(claims_scored['suspicion_score'] >= 50).sum()}")


=== DISTRIBUCIÓN DEL SCORE PONDERADO ===
count    10000.000000
mean        16.291000
std         11.473294
min          0.000000
25%          5.000000
50%         15.000000
75%         25.000000
max         70.000000
Name: suspicion_score, dtype: float64

Claims con score ≥ 30: 1526
Claims con score ≥ 50: 132


# 3. Exploración SQL de candidatos

In [19]:
# =========================
# Registro en DuckDB + query de screening
# =========================

con = duckdb.connect(str(DUCKDB_PATH))
con.register("claims_llm_base", claims_scored)

candidate_sql = """
SELECT
    transaction_id,
    insurance_type,
    claim_amount,
    premium_amount,
    ROUND(claim_to_premium_ratio, 2) AS ratio,
    incident_severity,
    days_report_delay,
    police_report_available,
    has_vendor,
    claim_status,
    suspicion_score
FROM claims_llm_base
WHERE suspicion_score >= 30
ORDER BY suspicion_score DESC, claim_amount DESC
LIMIT 30
"""

print("=== TOP 30 CLAIMS CANDIDATOS PARA AUDITORÍA LLM ===")
display(con.execute(candidate_sql).df())
con.close()


=== TOP 30 CLAIMS CANDIDATOS PARA AUDITORÍA LLM ===


,transaction_id,insurance_type,claim_amount,premium_amount,ratio,incident_severity,days_report_delay,police_report_available,has_vendor,claim_status,suspicion_score
0,TXN00009844,Life,90000.0,61.42,1465.32,Total Loss,3,0,0,A,70
1,TXN00004375,Life,69000.0,84.10,820.45,Total Loss,0,0,0,A,70
2,TXN00001573,Life,99000.0,88.21,1122.32,Total Loss,5,0,0,D,60
3,TXN00005959,Life,99000.0,96.53,1025.59,Total Loss,0,0,1,A,60
4,TXN00008898,Life,95000.0,55.82,1701.90,Total Loss,0,0,0,A,60
5,TXN00009986,Life,93000.0,97.20,956.79,Total Loss,3,0,1,A,60
6,TXN00003439,Life,93000.0,55.69,1669.96,Total Loss,5,0,0,A,60
7,TXN00000506,Life,86000.0,68.89,1248.37,Total Loss,0,0,1,A,60
8,TXN00001376,Life,85000.0,86.90,978.14,Total Loss,5,0,1,A,60
9,TXN00006111,Life,79000.0,73.59,1073.52,Major Loss,0,1,0,A,60


# 4. Selección de registros reales para la demo

In [20]:
# =========================
# Se eligen: 7 claims sospechosos (score alto) + 3 normal (score bajo)
# =========================

demo_suspicious = claims_scored.nlargest(7, "suspicion_score")
demo_normal = claims_scored[claims_scored["suspicion_score"] <= 5].sample(3, random_state=42)
demo_cases = pd.concat([demo_suspicious, demo_normal]).reset_index(drop=True)

demo_cols = ["transaction_id", "insurance_type", "claim_amount", "premium_amount",
             "incident_severity", "days_report_delay", "police_report_available",
             "has_vendor", "claim_status", "suspicion_score"]

print("=== CLAIMS SELECCIONADOS PARA LA DEMO ===")
display(demo_cases[[c for c in demo_cols if c in demo_cases.columns]])
print("\n→ Los primeros 7 son sospechosos (score alto). Los últimos 3 son normales (score bajo).")


=== CLAIMS SELECCIONADOS PARA LA DEMO ===


,transaction_id,insurance_type,claim_amount,premium_amount,incident_severity,days_report_delay,police_report_available,has_vendor,claim_status,suspicion_score
0,TXN00004375,Life,69000.0,84.10,Total Loss,0,0,0,A,70
1,TXN00009844,Life,90000.0,61.42,Total Loss,3,0,0,A,70
2,TXN00000014,Life,51000.0,64.16,Total Loss,5,0,0,A,60
3,TXN00000126,Life,36000.0,51.12,Total Loss,5,0,1,A,60
4,TXN00000271,Life,72000.0,82.70,Total Loss,1,0,0,A,60
5,TXN00000506,Life,86000.0,68.89,Total Loss,0,0,1,A,60
6,TXN00000673,Life,62000.0,74.65,Total Loss,5,0,0,A,60
7,TXN00000855,Property,27000.0,89.80,Major Loss,5,0,1,A,5
8,TXN00002985,Property,40000.0,142.35,Minor Loss,5,0,1,A,5
9,TXN00006105,Travel,2000.0,71.64,Total Loss,5,1,1,A,5



→ Los primeros 7 son sospechosos (score alto). Los últimos 3 son normales (score bajo).


# 5. Construcción del contexto y prompt

### Decisiones de diseño

**Contexto (JSON):** No se manda el row crudo. Se arma un dict compacto con identificadores, métricas, estado del claim, y la lista explícita de `risk_flags`. Esto mejora la calidad de la respuesta del LLM.

**System prompt:** Rol de auditor senior con restricciones claras (no acusar fraude, no inventar hechos, usar solo info provista).

**Output:** 5 secciones fijas para que la salida sea parseable y comparable entre claims.


In [21]:
# =========================
# Función de construcción de contexto
# =========================

def build_claim_context(row: pd.Series) -> dict:
    """Transforma un row del DataFrame en un contexto compacto para el LLM."""
    def safe(v):
        if isinstance(v, (np.integer,)): return int(v)
        if isinstance(v, (np.floating,)): return float(v)
        if pd.isna(v): return None
        return v
    
    # Detectar flags activas
    flag_cols = [c for c in row.index if c.startswith("flag_") and safe(row[c]) == 1]
    flag_descriptions = {
        "flag_high_ratio": "Ratio claim/premium extremadamente alto",
        "flag_same_day_report": "Reporte el mismo día del siniestro",
        "flag_total_loss_no_police": "Total Loss sin reporte policial",
        "flag_vendor_missing": "Sin vendor investigador asignado",
        "flag_cross_state": "Incidente en estado diferente al del cliente",
        "flag_night_high_severity": "Incidente nocturno con severidad alta",
        "flag_recent_policy": "Póliza con menos de 30 días al momento del siniestro",
        "flag_no_authority_severe": "Severidad alta sin autoridad contactada",
    }
    
    ratio_col = "claim_to_premium_ratio" if "claim_to_premium_ratio" in row.index else "loss_ratio"
    delay_col = "days_report_delay" if "days_report_delay" in row.index else "report_delay_days"
    
    return {
        "transaction_id": str(row.get("transaction_id", "N/A")),
        "insurance_type": str(row.get("insurance_type", "N/A")),
        "claim_amount": safe(row.get("claim_amount", 0)),
        "premium_amount": safe(row.get("premium_amount", 0)),
        "claim_to_premium_ratio": round(float(row.get(ratio_col, 0)), 2),
        "report_delay_days": safe(row.get(delay_col, 0)),
        "incident_severity": str(row.get("incident_severity", "N/A")),
        "any_injury": "Sí" if safe(row.get("any_injury", 0)) == 1 else "No",
        "police_report_available": "Sí" if safe(row.get("police_report_available", 0)) == 1 else "No",
        "has_vendor": "Sí" if safe(row.get("has_vendor", 0)) == 1 else "No",
        "authority_contacted": str(row.get("authority_contacted", "N/A")),
        "risk_segmentation": str(row.get("risk_segmentation", "N/A")),
        "age": safe(row.get("age", 0)),
        "claim_status": str(row.get("claim_status", "N/A")),
        "suspicion_score": safe(row.get("suspicion_score", 0)),
        "risk_flags": [flag_descriptions.get(f, f) for f in flag_cols],
    }

# Demo: mostrar contexto del primer claim sospechoso
sample_context = build_claim_context(demo_cases.iloc[0])
print("=== CONTEXTO GENERADO (claim sospechoso) ===")
print(json.dumps(sample_context, indent=2, ensure_ascii=False))


=== CONTEXTO GENERADO (claim sospechoso) ===
{
  "transaction_id": "TXN00004375",
  "insurance_type": "Life",
  "claim_amount": 69000.0,
  "premium_amount": 84.1,
  "claim_to_premium_ratio": 820.45,
  "report_delay_days": 0,
  "incident_severity": "Total Loss",
  "any_injury": "Sí",
  "police_report_available": "No",
  "has_vendor": "No",
  "authority_contacted": "Police",
  "risk_segmentation": "M",
  "age": 25,
  "claim_status": "A",
  "suspicion_score": 70,
  "risk_flags": [
    "Ratio claim/premium extremadamente alto",
    "Reporte el mismo día del siniestro",
    "Total Loss sin reporte policial",
    "Sin vendor investigador asignado",
    "Incidente en estado diferente al del cliente",
    "Incidente nocturno con severidad alta"
  ]
}


In [23]:
# =========================
# Prompt del auditor
# =========================

SYSTEM_PROMPT = dedent("""
    Sos un auditor senior de siniestros de una compañía aseguradora.
    Tu trabajo es analizar claims individuales y generar una nota de auditoría
    operativa para el equipo de siniestros.

    REGLAS:
    - Basá tu análisis SOLO en los datos provistos. No inventés información.
    - No acuses fraude directamente. Señalá riesgos y recomendá acciones.
    - No tomes decisión final de aprobación/rechazo, si no recomendá qué debería investigarse y da aviso sobre el caso al equipo de siniestros.
    - Sé conciso y directo. Máximo 250 palabras.

    CONTEXTO DEL PORTAFOLIO (benchmarks para comparación):
    - Loss ratio promedio del portafolio: 199x
    - Loss ratio promedio de Life: 754x (línea más expuesta)
    - Tasa de rechazo global: 5.03%
    - Tiempo promedio de reporte: 3.2 días
    - 32.5% de claims no tienen vendor investigador asignado
""").strip()


def build_audit_prompt(context: dict) -> tuple:
    """Devuelve (system_prompt, user_prompt) para el LLM."""
    user_prompt = dedent(f"""
        Analizá el siguiente claim y generá una nota de auditoría.

        DATOS DEL CLAIM:
        {json.dumps(context, indent=2, ensure_ascii=False)}

        FORMATO DE RESPUESTA (respetar estas 5 secciones):
        1. RESUMEN DEL CASO: [descripción breve del claim en 1-2 oraciones]
        2. SEÑALES DE RIESGO: [listá cada señal detectada con justificación]
        3. VACÍOS DE EVIDENCIA: [qué información falta o debería validarse]
        4. RECOMENDACIÓN OPERATIVA: [acción concreta sugerida y como operar en consecuencia]
        5. PRIORIDAD: [Alta / Media / Baja]
    """).strip()
    
    return SYSTEM_PROMPT, user_prompt

# Mostrar prompt generado
sys_p, usr_p = build_audit_prompt(sample_context)
print("===== SYSTEM PROMPT =====")
print(sys_p)
print("\n===== USER PROMPT =====")
print(usr_p)


===== SYSTEM PROMPT =====
Sos un auditor senior de siniestros de una compañía aseguradora.
Tu trabajo es analizar claims individuales y generar una nota de auditoría
operativa para el equipo de siniestros.

REGLAS:
- Basá tu análisis SOLO en los datos provistos. No inventés información.
- No acuses fraude directamente. Señalá riesgos y recomendá acciones.
- No tomes decisión final de aprobación/rechazo, si no recomendá qué debería investigarse y da aviso sobre el caso al equipo de siniestros.
- Sé conciso y directo. Máximo 250 palabras.

CONTEXTO DEL PORTAFOLIO (benchmarks para comparación):
- Loss ratio promedio del portafolio: 199x
- Loss ratio promedio de Life: 754x (línea más expuesta)
- Tasa de rechazo global: 5.03%
- Tiempo promedio de reporte: 3.2 días
- 32.5% de claims no tienen vendor investigador asignado

===== USER PROMPT =====
Analizá el siguiente claim y generá una nota de auditoría.

        DATOS DEL CLAIM:
        {
  "transaction_id": "TXN00004375",
  "insurance_type"

# 6. Demo offline reproducible

Para que el notebook sea **100% ejecutable sin API**, se incluye una simulación offline que replica la estructura de salida esperada.

Esta función **no reemplaza al LLM**, pero permite validar la estructura del contexto, la calidad de las flags, y el formato de salida.


In [26]:
# =========================
# Simulación offline (fallback sin API)
# =========================

def mock_llm_audit(context: dict) -> str:
    """Genera una nota de auditoría simulada basada en reglas.
    En producción se reemplaza por la llamada real al LLM.
    """
    flags = context.get("risk_flags", [])
    n = len(flags)
    txn = context["transaction_id"]
    ins = context["insurance_type"]
    amt = context["claim_amount"]
    ratio = context["claim_to_premium_ratio"]
    score = context["suspicion_score"]
    
    # Prioridad basada en score
    if score >= 40:
        priority = "Alta"
        rec = "Derivar inmediatamente a investigación con asignación de vendor."
    elif score >= 20:
        priority = "Media"
        rec = "Revisar documentación y validar con el asegurado antes de aprobar."
    else:
        priority = "Baja"
        rec = "Procesar normalmente. No se detectaron señales de alerta significativas."
    
    flags_text = "\n".join(f"  • {f}" for f in flags) if flags else "  Ninguna detectada."
    
    # Vacíos de evidencia
    gaps = []
    if context["police_report_available"] == "No":
        gaps.append("Falta reporte policial para un siniestro de esta severidad.")
    if context["has_vendor"] == "No":
        gaps.append("No hay vendor investigador asignado al caso.")
    if context["authority_contacted"] in ["None", "N/A", None]:
        gaps.append("No se registra contacto con autoridad competente.")
    gaps_text = "\n".join(f"  • {g}" for g in gaps) if gaps else "  Sin vacíos críticos detectados."
    
    return (
        f"1. RESUMEN DEL CASO:\n"
        f"   Claim {txn} de tipo {ins} por ${amt:,.0f} con ratio claim/prima de {ratio:.0f}x. "
        f"Score de sospecha: {score} puntos.\n\n"
        f"2. SEÑALES DE RIESGO:\n{flags_text}\n\n"
        f"3. VACÍOS DE EVIDENCIA:\n{gaps_text}\n\n"
        f"4. RECOMENDACIÓN OPERATIVA:\n   {rec}\n\n"
        f"5. PRIORIDAD: {priority}"
    )

# Demo offline con el primer claim
print("=== DEMO OFFLINE — Nota de auditoría simulada ===\n")
print(mock_llm_audit(sample_context))


=== DEMO OFFLINE — Nota de auditoría simulada ===

1. RESUMEN DEL CASO:
   Claim TXN00004375 de tipo Life por $69,000 con ratio claim/prima de 820x. Score de sospecha: 70 puntos.

2. SEÑALES DE RIESGO:
  • Ratio claim/premium extremadamente alto
  • Reporte el mismo día del siniestro
  • Total Loss sin reporte policial
  • Sin vendor investigador asignado
  • Incidente en estado diferente al del cliente
  • Incidente nocturno con severidad alta

3. VACÍOS DE EVIDENCIA:
  • Falta reporte policial para un siniestro de esta severidad.
  • No hay vendor investigador asignado al caso.

4. RECOMENDACIÓN OPERATIVA:
   Derivar inmediatamente a investigación con asignación de vendor.

5. PRIORIDAD: Alta


# 7. Demo con API real de OpenAI (opcional)

Si existe la variable de entorno `OPENAI_API_KEY`, el notebook ejecuta una llamada real al modelo.

### Setup
```powershell
$env:OPENAI_API_KEY="sk-..."
```
O archivo `.env`:
```
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
```

Esta celda **no rompe el notebook** si no hay credenciales.


In [27]:
# =========================
# Función de llamada real a OpenAI
# =========================

def run_openai_audit(context: dict, model: str = MODEL_NAME) -> str:
    """Ejecuta la auditoría usando la API de OpenAI."""
    system_prompt, user_prompt = build_audit_prompt(context)
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=500,
        temperature=0.2,  # Baja temperatura para consistencia
    )
    return response.choices[0].message.content


def audit_claim(context: dict) -> str:
    """Wrapper: usa API real si disponible, sino fallback offline."""
    if USE_REAL_API:
        return run_openai_audit(context)
    else:
        return mock_llm_audit(context)

# Ejecutar sobre el primer claim
if USE_REAL_API:
    print("=== DEMO CON API REAL (OpenAI) ===\n")
    print(audit_claim(sample_context))
else:
    print("Sin OPENAI_API_KEY configurada. Se mantiene la demo offline.")
    print("El código está listo para ejecutar con credenciales.")


=== DEMO CON API REAL (OpenAI) ===

1. RESUMEN DEL CASO:  
El claim TXN00004375 corresponde a un siniestro de tipo Life, con un monto reclamado de $69,000 y una prima de $84.1, resultando en un ratio de reclamo a prima extremadamente alto de 820.45. El incidente se reportó el mismo día, pero no se cuenta con un informe policial.

2. SEÑALES DE RIESGO:  
- **Ratio claim/premium extremadamente alto**: Indica un potencial riesgo de abuso o irregularidad.  
- **Reporte el mismo día del siniestro**: Puede ser una señal de manipulación de fechas.  
- **Total Loss sin reporte policial**: La falta de un informe policial puede dificultar la verificación de los hechos.  
- **Sin vendor investigador asignado**: Aumenta la dificultad para validar la reclamación.  
- **Incidente en estado diferente al del cliente**: Esto podría indicar inconsistencias en la narrativa del siniestro.  
- **Incidente nocturno con severidad alta**: Aumenta la complejidad del caso y puede requerir mayor escrutinio.

3. 

# 8. Lote de demo sobre múltiples claims

Se ejecuta la auditoría sobre los 4 claims seleccionados para validar que el sistema diferencia correctamente entre claims sospechosos y normales.


In [28]:
# =========================
# Auditoría batch sobre los 4 claims demo
# =========================

demo_contexts = [build_claim_context(row) for _, row in demo_cases.iterrows()]

print("=" * 80)
print(f"AUDITORÍA AUTOMÁTICA — {'API OpenAI' if USE_REAL_API else 'Demo Offline'}")
print("=" * 80)

batch_results = []

for i, ctx in enumerate(demo_contexts):
    print(f"\n{'─' * 70}")
    print(f"CLAIM {i+1}: {ctx['transaction_id']} | {ctx['insurance_type']} | "
          f"${ctx['claim_amount']:,.0f} | Score: {ctx['suspicion_score']} | Status: {ctx['claim_status']}")
    print(f"{'─' * 70}")
    
    output = audit_claim(ctx)
    print(output)
    
    # Extraer prioridad de la salida
    priority_line = [l for l in output.split("\n") if "PRIORIDAD" in l.upper()]
    priority = priority_line[0].split(":")[-1].strip() if priority_line else "N/A"
    
    batch_results.append({
        "transaction_id": ctx["transaction_id"],
        "insurance_type": ctx["insurance_type"],
        "claim_amount": ctx["claim_amount"],
        "suspicion_score": ctx["suspicion_score"],
        "status_real": ctx["claim_status"],
        "prioridad_auditor": priority,
    })


AUDITORÍA AUTOMÁTICA — API OpenAI

──────────────────────────────────────────────────────────────────────
CLAIM 1: TXN00004375 | Life | $69,000 | Score: 70 | Status: A
──────────────────────────────────────────────────────────────────────
1. RESUMEN DEL CASO:  
El claim TXN00004375 corresponde a un siniestro de tipo Life, con un monto reclamado de $69,000 y una prima de $84.1, resultando en un ratio de reclamo a prima extremadamente alto de 820.45. El incidente se reportó el mismo día, pero no se cuenta con un informe policial.

2. SEÑALES DE RIESGO:  
- **Ratio claim/premium extremadamente alto**: Indica un riesgo significativo, ya que supera ampliamente los benchmarks del portafolio.  
- **Reporte el mismo día del siniestro**: Puede generar dudas sobre la veracidad del reclamo.  
- **Total Loss sin reporte policial**: La falta de un informe policial puede dificultar la verificación de los hechos.  
- **Sin vendor investigador asignado**: Aumenta el riesgo de no tener una evaluación o

In [29]:
# =========================
# Resumen de la auditoría batch
# =========================

results_df = pd.DataFrame(batch_results)
print("\n" + "=" * 80)
print("RESUMEN DE AUDITORÍA")
print("=" * 80)
display(results_df)

print("\n→ Claims sospechosos → prioridad Alta.")
print("→ Claim normal → prioridad Baja.")
print("→ Todos fueron aprobados ('A') en la realidad, pero el auditor LLM")
print("  hubiera derivado los sospechosos a investigación adicional.")



RESUMEN DE AUDITORÍA


,transaction_id,insurance_type,claim_amount,suspicion_score,status_real,prioridad_auditor
0,TXN00004375,Life,69000.0,70,A,Alta
1,TXN00009844,Life,90000.0,70,A,
2,TXN00000014,Life,51000.0,60,A,
3,TXN00000126,Life,36000.0,60,A,
4,TXN00000271,Life,72000.0,60,A,
5,TXN00000506,Life,86000.0,60,A,
6,TXN00000673,Life,62000.0,60,A,
7,TXN00000855,Property,27000.0,5,A,Alta
8,TXN00002985,Property,40000.0,5,A,
9,TXN00006105,Travel,2000.0,5,A,



→ Claims sospechosos → prioridad Alta.
→ Claim normal → prioridad Baja.
→ Todos fueron aprobados ('A') en la realidad, pero el auditor LLM
  hubiera derivado los sospechosos a investigación adicional.


# 9. Persistencia de outputs

In [31]:
# =========================
# Exportar contextos y outputs para trazabilidad
# =========================

# Contextos JSON (input del LLM)
with open(OUTPUT_DIR / "llm_demo_contexts.json", "w", encoding="utf-8") as f:
    json.dump(demo_contexts, f, ensure_ascii=False, indent=2)

# Resultados de la auditoría
results_df.to_csv(OUTPUT_DIR / "llm_audit_results.csv", index=False)

print(f"\n Contextos exportados: {OUTPUT_DIR / 'llm_demo_contexts.json'}")
print(f"\n Resultados exportados: {OUTPUT_DIR / 'llm_audit_results.csv'}")



 Contextos exportados: c:\Users\Ash\Desktop\Proyectos\Challenge-TEKNE\Desafio_3_IA\outputs_llm\llm_demo_contexts.json

 Resultados exportados: c:\Users\Ash\Desktop\Proyectos\Challenge-TEKNE\Desafio_3_IA\outputs_llm\llm_audit_results.csv


# 10. Valor de negocio

## Qué proceso mejora
El proceso de **triage y revisión inicial de siniestros**. No automatiza la decisión final, pero ayuda a:
- ordenar la cola de revisión por prioridad,
- reducir tiempo de lectura y análisis inicial,
- uniformar el criterio narrativo entre analistas,
- dejar trazabilidad de las señales detectadas.

## Cómo se mediría el impacto

| Métrica | Cómo se mide | Target |
|---------|-------------|--------|
| **Tiempo de triaje** | Minutos promedio por claim | De ~15 min a ~5 min |
| **Tasa de detección** | % de fraudes detectados antes de pago | Subir del 5% actual |
| **Cobertura de investigación** | % de claims de alto riesgo con vendor | Del 68% al 90%+ |
| **Costo por auditoría** | USD por claim procesado | ~$0.002 (LLM) vs ~$15 (humano) |
| **Adopción** | % de analistas que usan las notas | Target: >80% |

## Estimación de costos (OpenAI gpt-4o-mini)

| Concepto | Estimación |
|----------|-----------|
| Tokens input por claim | ~500 |
| Tokens output por claim | ~300 |
| Costo por claim | ~$0.0003 |
| Claims con score ≥ 30/día (estimado) | ~20-50 |
| Costo mensual estimado | ~$0.20 - $0.50 |

## Riesgos y controles
- Revisión humana obligatoria antes de cualquier acción.
- Prompts con restricciones claras (no acusar fraude, no decidir).
- Monitorear alucinaciones y tasa de falsos positivos.
- Registrar inputs/outputs para auditoría interna.
- No usar como motor de decisión automática.


# 11. Resumen del Punto 3

## Caso de uso
**Auditor Automático de Siniestros Sospechosos** — un LLM recibe datos estructurados + flags de riesgo y genera una nota de auditoría operativa.

## Flujo implementado
1. Score heurístico ponderado (8 flags, pesos diferenciados).
2. Screening SQL de candidatos.
3. Construcción de contexto compacto (JSON).
4. Prompt con 5 secciones fijas (resumen, señales, vacíos, recomendación, prioridad).
5. Demo sobre 4 claims reales (3 sospechosos + 1 normal).
6. Persistencia de inputs/outputs.

## Entregables
- Prompt completo y funcional.
- Código Python reproducible con SDK de OpenAI.
- Demo offline como fallback (100% ejecutable sin API).
- Contextos y resultados exportados en JSON/CSV.
- No requiere deploy ni UI.

## En una siguiente iteración
- Agregar salidas estructuradas JSON para parseo automático.
- Evaluar varios modelos/prompts con métricas de calidad.
- Sumar feedback humano para calibración.
- Acoplar a cola real de revisión de siniestros.

---
